<strong><ins style="color:lightblue"><span style="color:lightblue;font-size:50px"> Purchase Data Customer Segmentation Analysis</strong>
<span style="font-size:15px"><br> Collect and Agregate Purchases to Customers
<br> create Customer Cluster Groups
<br> Using Customer Cluster Groups Start to Create a product propensity model
<br> online data source "https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset?resource=download" 

In [ ]:
#imports
import datetime as dt
print(f"Job Start: {dt.datetime.now()}")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

<strong><ins style="color:green"><span style= "color:green;font-size:35px"> Data Collection

In [3]:
print("starting data imports...")

catergory_tree = 'C:/Users/zicza/Downloads/customer_segmentation_analysis/category_tree.csv'
events = 'C:/Users/zicza/Downloads/customer_segmentation_analysis/events.csv'
item_properties_pt1 = 'C:/Users/zicza/Downloads/customer_segmentation_analysis/item_properties_part1.csv'
item_properties_pt2 = 'C:/Users/zicza/Downloads/customer_segmentation_analysis/item_properties_part2.csv'

#read in the data
df_catergory_tree = pd.read_csv(catergory_tree)
df_events = pd.read_csv(events)
df_item_properties_pt1 = pd.read_csv(item_properties_pt1)
df_item_properties_pt2 = pd.read_csv(item_properties_pt2)

print(f"df_categories: {df_catergory_tree.columns}")
print(f"df_events: {df_events.columns}")
print(f"df_item_properties_1: {df_item_properties_pt1.columns}")
print(f"df_item_properties_1: {df_item_properties_pt2.columns}")

#convert to data frames
df_catergory_tree = pd.DataFrame(df_catergory_tree)
df_events = pd.DataFrame(df_events)
df_item_properties_pt1 = pd.DataFrame(df_item_properties_pt1)
df_item_properties_pt2 = pd.DataFrame(df_item_properties_pt2)

df_item_properties = pd.concat([df_item_properties_pt1, df_item_properties_pt2])

print("Data Imports Complete")

df_categories: Index(['categoryid', 'parentid'], dtype='object')
df_events: Index(['timestamp', 'visitorid', 'event', 'itemid', 'transactionid'], dtype='object')
df_item_properties_1: Index(['timestamp', 'itemid', 'property', 'value'], dtype='object')
df_item_properties_1: Index(['timestamp', 'itemid', 'property', 'value'], dtype='object')
Data Imports Complete


<strong><ins style="color:orange"><span style="color:orange;font-size:25px"> Data Clean-up

In [4]:
#df_events_clean-up

print("Cleaning up df_events...")
print(len(df_events))
df_events['timestamp'] = pd.to_datetime(df_events['timestamp'],unit='ms')
print("timeste converted")
df_events['event'] = df_events['event'].astype(str)
print("event type converted")
df_events = df_events.sort_values(by=['visitorid', 'timestamp'])
print("df_events sorted")
df_events['first_event'] = df_events.groupby('visitorid')['timestamp'].transform('first')
print("first event time added")
df_events['last_event'] = df_events.groupby('visitorid')['timestamp'].transform('last')
print("last event time added")
df_events['session_duration'] = df_events['last_event'] - df_events['first_event']
print("session duration added")
df_events['products_viewed'] = df_events.groupby('visitorid')['itemid'].transform(lambda x: (x=='view').sum())
print("count product viewings added")
df_events['products_added_to_basket'] = df_events.groupby('visitorid')['event'].transform(lambda x: (x=='addtocart').sum())
print("count products in cart added")
df_events['products_purchased'] = df_events.groupby('visitorid')['event'].transform(lambda x: (x=='transaction').sum())
print("count product purchases added")

print("Cleaning up df_events complete.")

print("cleaning up df_item_properties...")
df_item_properties['timestamp'] = pd.to_datetime(df_item_properties['timestamp'],unit='ms')
df_item_properties['property'] = df_item_properties['property'].astype(str)

print("cleaning up df_item_properties complete.")


Cleaning up df_events...
2756101
timeste converted
event type converted
df_events sorted
first event time added
last event time added
session duration added
count product viewings added
count products in cart added
count product purchases added
Cleaning up df_events complete.
cleaning up df_item_properties...
cleaning up df_item_properties complete.


<strong><ins style="color:orange"><span style="color:orange;font-size:25px"> Event Facts per vistior

In [9]:
print(df_events.head())

df_events_agregate = df_events[['visitorid', 'session_duration', 'products_viewed', 'products_added_to_basket', 'products_purchased']].drop_duplicates()
print(df_events_agregate.head())

                      timestamp  visitorid event  itemid  transactionid  \
1361687 2015-09-11 20:49:49.439          0  view  285930            NaN   
1367212 2015-09-11 20:52:39.591          0  view  357564            NaN   
1367342 2015-09-11 20:55:17.175          0  view   67045            NaN   
830385  2015-08-13 17:46:06.444          1  view   72028            NaN   
742616  2015-08-07 17:51:44.567          2  view  325215            NaN   

                    first_event              last_event  \
1361687 2015-09-11 20:49:49.439 2015-09-11 20:55:17.175   
1367212 2015-09-11 20:49:49.439 2015-09-11 20:55:17.175   
1367342 2015-09-11 20:49:49.439 2015-09-11 20:55:17.175   
830385  2015-08-13 17:46:06.444 2015-08-13 17:46:06.444   
742616  2015-08-07 17:51:44.567 2015-08-07 18:20:57.845   

              session_duration  products_viewed  products_added_to_basket  \
1361687 0 days 00:05:27.736000                0                         0   
1367212 0 days 00:05:27.736000          